# NB13 — Manuscript Figures

Renders all data figures referenced in the manuscript by reading the CSVs and OOF arrays saved by upstream notebooks. No new model runs. The five schematic figures (1A, 1B, 2A, 2B) are hand-drawn and not produced here.

Produces:
- **Figure 1C** — computational efficiency comparison vs UNI/Virchow/Virchow2-Giant (numbers from manuscript Table 3)
- **Figure 3A** — per-cancer F1 across 31 TCGA cancer types (NB07)
- **Figure 3B** — per-cancer AUROC across 31 TCGA cancer types (NB07)
- **Figure 3C** — F1 stratified by organ system (NB07)
- **Figure 3D** — accuracy vs test-set size scatter with regression line (NB07)
- **Figure 4A** — CAMELYON16 ROC curve, OpenSlideFM vs UNI/Virchow reference lines (NB09A)
- **Figure 4B** — PANDA F1/precision/recall per ISUP grade (NB11)
- **Figure 4C** — CAMELYON17 per-center quadratic-weighted κ (NB09)
- **Figure 4D** — CAMELYON17 stage transition matrix (NB09)
- **Figure 4E** — TCGA 10-class per-cancer F1, OpenSlideFM vs UNI2-h (NB07 + UNI2-h CSV)
- **Figure 4F** — PANDA within-Karolinska vs within-Radboud κ (NB12)
- **Supp Fig 1** — UMAP of TCGA 768-d slide embeddings, colored by cancer type and prediction confidence (NB07 + NB08)

In [ ]:
import os, sys, json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, auc, classification_report

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
FIG_DIR = WORKSPACE / 'figures' / 'manuscript'
FIG_DIR.mkdir(parents=True, exist_ok=True)

DPI = 300
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

ORGAN_SYSTEM = {
    'BRCA': 'Breast', 'CESC': 'Reproductive', 'OV': 'Reproductive', 'UCEC': 'Reproductive',
    'UCS': 'Reproductive', 'PRAD': 'Genitourinary', 'KIRC': 'Genitourinary',
    'KIRP': 'Genitourinary', 'KICH': 'Genitourinary', 'BLCA': 'Genitourinary',
    'TGCT': 'Genitourinary', 'LUAD': 'Respiratory', 'LUSC': 'Respiratory',
    'MESO': 'Respiratory', 'COAD': 'Gastrointestinal', 'READ': 'Gastrointestinal',
    'STAD': 'Gastrointestinal', 'ESCA': 'Gastrointestinal', 'LIHC': 'Gastrointestinal',
    'CHOL': 'Gastrointestinal', 'PAAD': 'Gastrointestinal', 'HNSC': 'Head/Neck',
    'THCA': 'Endocrine', 'ACC': 'Endocrine', 'PCPG': 'Endocrine', 'LGG': 'CNS',
    'GBM': 'CNS', 'SKCM': 'Skin', 'UVM': 'Skin', 'DLBC': 'Hematologic',
    'THYM': 'Hematologic', 'SARC': 'Soft tissue',
}

def save_fig(fig, name):
    out = FIG_DIR / name
    fig.savefig(out, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f'  [OK] {out}')

# Figure 1C: computational efficiency comparison from Table 3 numbers
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
models = ['OpenSlideFM', 'UNI', 'Virchow', 'Virchow2-G']
params_M = [71, 307, 632, 1850]
vram_GB  = [24, 40, 40, 80]
colors = ['#2E86AB', '#999999', '#999999', '#999999']
ax = axes[0]
bars = ax.bar(models, params_M, color=colors, edgecolor='black')
ax.set_ylabel('Parameters (M)')
ax.set_title('Model size')
ax.set_yscale('log')
for b, v in zip(bars, params_M):
    ax.text(b.get_x() + b.get_width()/2, v * 1.05, f'{v}M', ha='center', va='bottom', fontsize=9)
ax = axes[1]
bars = ax.bar(models, vram_GB, color=colors, edgecolor='black')
ax.set_ylabel('GPU memory (GB)')
ax.set_title('Hardware requirement')
for b, v in zip(bars, vram_GB):
    ax.text(b.get_x() + b.get_width()/2, v + 1, f'{v} GB', ha='center', va='bottom', fontsize=9)
for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
fig.suptitle('Figure 1C: Computational efficiency vs published foundation models', y=1.02)
fig.tight_layout()
save_fig(fig, 'fig1C_computational_efficiency.png')

# TCGA evaluation outputs from NB07
tcga_dir = WORKSPACE / 'results' / 'tcga_baseline_evaluation'
per_class_csv = tcga_dir / 'per_class_metrics.csv'
oof_npz = tcga_dir / 'oof.npz'

if per_class_csv.exists():
    per_class = pd.read_csv(per_class_csv, index_col=0)
    cancer_rows = per_class.drop(
        index=[r for r in per_class.index if r in ('accuracy', 'macro avg', 'weighted avg')],
        errors='ignore',
    )
    cancer_rows['cancer'] = cancer_rows.index
    cancer_rows = cancer_rows.sort_values('f1-score', ascending=True)

    # Figure 3A: per-cancer F1
    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(cancer_rows['cancer'], cancer_rows['f1-score'], color='#2E86AB', edgecolor='black')
    ax.set_xlabel('F1-score')
    ax.set_xlim(0, 1.0)
    ax.axvline(cancer_rows['f1-score'].mean(), ls='--', color='red', alpha=0.6,
               label=f'mean F1 = {cancer_rows["f1-score"].mean():.3f}')
    ax.set_title('Figure 3A: Per-cancer F1-scores across 31 TCGA cancer types')
    ax.legend()
    fig.tight_layout()
    save_fig(fig, 'fig3A_per_cancer_f1.png')

    # Figure 3C: F1 by organ system
    cancer_rows['organ'] = cancer_rows['cancer'].map(ORGAN_SYSTEM).fillna('Other')
    organ_summary = cancer_rows.groupby('organ').agg(
        f1_mean=('f1-score', 'mean'),
        f1_std=('f1-score', 'std'),
        n_types=('cancer', 'count'),
    ).reset_index().sort_values('f1_mean', ascending=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(organ_summary['organ'], organ_summary['f1_mean'],
           yerr=organ_summary['f1_std'].fillna(0), capsize=4,
           color='#2E86AB', edgecolor='black')
    for i, (mean, n) in enumerate(zip(organ_summary['f1_mean'], organ_summary['n_types'])):
        ax.text(i, mean + 0.02, f'n={n}', ha='center', fontsize=8)
    ax.set_ylabel('Mean F1-score')
    ax.set_ylim(0, 1.0)
    ax.set_title('Figure 3C: F1 stratified by organ system')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    fig.tight_layout()
    save_fig(fig, 'fig3C_organ_system_f1.png')

    # Figure 3D: accuracy vs test-set size
    if 'support' in cancer_rows.columns:
        fig, ax = plt.subplots(figsize=(7, 5))
        x = cancer_rows['support'].astype(float).values
        y = cancer_rows['f1-score'].astype(float).values
        ax.scatter(x, y, alpha=0.7, color='#2E86AB', edgecolor='black', s=50)
        for cx, cy, lbl in zip(x, y, cancer_rows['cancer']):
            ax.annotate(lbl, (cx, cy), fontsize=7, alpha=0.7)
        if len(x) >= 3:
            r = np.corrcoef(np.log10(x.clip(min=1)), y)[0, 1]
            slope, intercept = np.polyfit(np.log10(x.clip(min=1)), y, 1)
            xs = np.linspace(np.log10(max(x.min(), 1)), np.log10(x.max()), 100)
            ax.plot(10**xs, slope*xs + intercept, '--', color='red', alpha=0.6,
                    label=f'Pearson r = {r:.3f}')
            ax.legend()
        ax.set_xscale('log')
        ax.set_xlabel('Test set size (patients per class)')
        ax.set_ylabel('F1-score')
        ax.set_title('Figure 3D: Performance vs test-set size')
        fig.tight_layout()
        save_fig(fig, 'fig3D_acc_vs_n.png')
else:
    print(f'  [SKIP] {per_class_csv} not found; run NB07 first')

# Figure 3B: per-cancer AUROC (needs OOF probabilities + classes from NB07)
if oof_npz.exists():
    npz = np.load(oof_npz, allow_pickle=True)
    y_oof = npz['y']
    probs_oof = npz['oof_probs']
    classes = list(npz['classes'])
    from sklearn.metrics import roc_auc_score
    aucs = []
    for k, cname in enumerate(classes):
        y_bin = (y_oof == k).astype(int)
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin): continue
        try:
            aucs.append((cname, float(roc_auc_score(y_bin, probs_oof[:, k]))))
        except Exception:
            pass
    aucs = sorted(aucs, key=lambda r: r[1])
    fig, ax = plt.subplots(figsize=(10, 8))
    names = [r[0] for r in aucs]; vals = [r[1] for r in aucs]
    ax.barh(names, vals, color='#A23B72', edgecolor='black')
    ax.set_xlabel('AUROC (one-vs-rest)')
    ax.set_xlim(0.5, 1.0)
    ax.axvline(np.mean(vals), ls='--', color='red', alpha=0.6,
               label=f'mean AUROC = {np.mean(vals):.3f}')
    ax.set_title('Figure 3B: Per-cancer AUROC across 31 TCGA cancer types')
    ax.legend()
    fig.tight_layout()
    save_fig(fig, 'fig3B_per_cancer_auroc.png')
else:
    print(f'  [SKIP] {oof_npz} not found; run NB07 first')

# Figure 4A: CAMELYON16 ROC + UNI/Virchow reference lines
cam16_oof = WORKSPACE / 'results' / 'cam16_eval' / 'cam16_oof.csv'
cam16_json = WORKSPACE / 'results' / 'cam16_eval' / 'cam16_results.json'
if cam16_oof.exists():
    df16 = pd.read_csv(cam16_oof)
    fpr, tpr, _ = roc_curve(df16['y_true'], df16['p_tumor'])
    osfm_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, color='#2E86AB', lw=2, label=f'OpenSlideFM (AUROC = {osfm_auc:.3f})')
    ax.axhline(0.795, ls='--', color='gray', alpha=0.7, label='UNI (0.795)')
    ax.axhline(0.812, ls='--', color='black', alpha=0.7, label='Virchow (0.812)')
    ax.plot([0, 1], [0, 1], ls=':', color='red', alpha=0.5, label='Random')
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title('Figure 4A: CAMELYON16 metastasis detection ROC')
    ax.legend(loc='lower right')
    fig.tight_layout()
    save_fig(fig, 'fig4A_cam16_roc.png')
else:
    print(f'  [SKIP] {cam16_oof} not found; run NB09A first')

# Figure 4B: PANDA F1/precision/recall per ISUP grade
panda_oof = WORKSPACE / 'results' / 'panda_mil' / 'oof_ensemble.csv'
if panda_oof.exists():
    dfp = pd.read_csv(panda_oof)
    grades = sorted(dfp['true_isup'].unique())
    f1s = []; precs = []; recs = []
    for g in grades:
        f1s.append(f1_score(dfp['true_isup'] == g, dfp['pred_isup'] == g))
        precs.append(precision_score(dfp['true_isup'] == g, dfp['pred_isup'] == g, zero_division=0))
        recs.append(recall_score(dfp['true_isup'] == g, dfp['pred_isup'] == g, zero_division=0))
    grade_labels = ['Benign', '<3+3', '3+4', '4+3', '4+4 or 3+5', '>4+5']
    grade_labels = grade_labels[:len(grades)]
    fig, ax = plt.subplots(figsize=(9, 5))
    width = 0.27
    pos = np.arange(len(grades))
    ax.bar(pos - width, precs, width, color='#A23B72', edgecolor='black', label='Precision')
    ax.bar(pos,         recs,  width, color='#F18F01', edgecolor='black', label='Recall')
    ax.bar(pos + width, f1s,   width, color='#2E86AB', edgecolor='black', label='F1')
    ax.set_xticks(pos); ax.set_xticklabels(grade_labels)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Score')
    ax.set_title('Figure 4B: PANDA per-ISUP-grade performance')
    ax.legend()
    fig.tight_layout()
    save_fig(fig, 'fig4B_panda_per_grade.png')
else:
    print(f'  [SKIP] {panda_oof} not found; run NB11 first')

# Figure 4C: CAMELYON17 per-center kappa
per_center_csv = WORKSPACE / 'results' / 'cam17_pn_eval' / 'ablations' / 'per_center_kappa.csv'
if per_center_csv.exists():
    dfc = pd.read_csv(per_center_csv).sort_values('kappa_qw', ascending=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#2E86AB' if v >= 0 else '#C73E1D' for v in dfc['kappa_qw']]
    bars = ax.bar(dfc['center'], dfc['kappa_qw'], color=colors, edgecolor='black')
    ax.axhline(0, color='black', lw=0.8)
    ax.axhline(dfc['kappa_qw'].mean(), ls='--', color='red', alpha=0.6,
               label=f'overall κ = {dfc["kappa_qw"].mean():.3f}')
    ax.set_ylabel('Quadratic-weighted κ')
    ax.set_title('Figure 4C: CAMELYON17 per-center generalization (LOCO-CV)')
    ax.legend()
    fig.tight_layout()
    save_fig(fig, 'fig4C_cam17_per_center.png')
else:
    print(f'  [SKIP] {per_center_csv} not found; run NB09 first')

# Figure 4D: CAMELYON17 stage transition matrix
trans_csv = WORKSPACE / 'results' / 'cam17_pn_eval' / 'ablations' / 'transition_matrix_normalized.csv'
if trans_csv.exists():
    dft = pd.read_csv(trans_csv, index_col=0)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(dft.values, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(dft.columns))); ax.set_xticklabels(dft.columns)
    ax.set_yticks(range(len(dft.index))); ax.set_yticklabels(dft.index)
    ax.set_xlabel('Predicted stage')
    ax.set_ylabel('True stage')
    for i in range(dft.shape[0]):
        for j in range(dft.shape[1]):
            v = dft.values[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.5 else 'black', fontsize=11)
    ax.set_title('Figure 4D: CAMELYON17 stage transition matrix (row-normalized)')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    save_fig(fig, 'fig4D_cam17_transitions.png')
else:
    print(f'  [SKIP] {trans_csv} not found; run NB09 first')

# Figure 4E: TCGA 10-class OpenSlideFM vs UNI2-h (requires uni2h CSV at known path)
uni2h_csv = WORKSPACE / 'results' / 'tcga_baseline_evaluation' / 'uni2h_per_class_metrics.csv'
TEN = ['ACC', 'BRCA-IDC', 'COAD', 'DLBC', 'GBM', 'HNSC', 'KIRC', 'LUAD', 'SKCM', 'UCEC']
if per_class_csv.exists() and uni2h_csv.exists():
    per_class = pd.read_csv(per_class_csv, index_col=0)
    uni2h = pd.read_csv(uni2h_csv, index_col=0)
    rows = []
    for code in TEN:
        if code in per_class.index and code in uni2h.index:
            rows.append({
                'cancer': code,
                'OpenSlideFM': float(per_class.loc[code, 'f1-score']),
                'UNI2-h': float(uni2h.loc[code, 'f1-score']),
            })
    df_cmp = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(9, 5))
    width = 0.4
    pos = np.arange(len(df_cmp))
    ax.bar(pos - width/2, df_cmp['OpenSlideFM'], width, color='#2E86AB', edgecolor='black', label='OpenSlideFM')
    ax.bar(pos + width/2, df_cmp['UNI2-h'],      width, color='#999999', edgecolor='black', label='UNI2-h')
    for i in range(len(df_cmp)):
        d = df_cmp['UNI2-h'].iloc[i] - df_cmp['OpenSlideFM'].iloc[i]
        if abs(d) >= 0.04:
            ymax = max(df_cmp['OpenSlideFM'].iloc[i], df_cmp['UNI2-h'].iloc[i]) + 0.02
            ax.text(i, ymax, f'Δ={d:+.2f}', ha='center', fontsize=8, color='red')
    ax.set_xticks(pos); ax.set_xticklabels(df_cmp['cancer'], rotation=30, ha='right')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('F1-score')
    ax.set_title('Figure 4E: TCGA 10-class per-cancer F1, OpenSlideFM vs UNI2-h')
    ax.legend()
    fig.tight_layout()
    save_fig(fig, 'fig4E_tcga10_osfm_vs_uni2h.png')
else:
    print(f'  [SKIP] Figure 4E needs {uni2h_csv} (UNI2-h baseline not computed in this pipeline)')

# Figure 4F: PANDA Karolinska vs Radboud kappa
panda_prov = WORKSPACE / 'results' / 'panda_mil' / 'metrics_auc_by_provider.csv'
if panda_prov.exists():
    dfprov = pd.read_csv(panda_prov)
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(dfprov['provider'], dfprov['macro_auroc_ovr'],
                  color=['#2E86AB', '#A23B72'], edgecolor='black')
    for b, v, n in zip(bars, dfprov['macro_auroc_ovr'], dfprov['n']):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}\n(n={n})',
                ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Macro AUROC (OvR)')
    ax.set_title('Figure 4F: PANDA cross-provider robustness')
    fig.tight_layout()
    save_fig(fig, 'fig4F_panda_cross_provider.png')
else:
    print(f'  [SKIP] {panda_prov} not found; run NB12 first')

# Supplementary Figure 1: TCGA UMAP
emb_dir = WORKSPACE / 'embeddings'
manifest_csv = WORKSPACE / 'manifests' / 'manifest_tcga.csv'
if oof_npz.exists() and manifest_csv.exists():
    try:
        import umap
        HAS_UMAP = True
    except Exception:
        try:
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn', '--break-system-packages'])
            import umap
            HAS_UMAP = True
        except Exception:
            HAS_UMAP = False
    if HAS_UMAP:
        npz = np.load(oof_npz, allow_pickle=True)
        y_oof = npz['y']
        probs_oof = npz['oof_probs']
        classes = list(npz['classes'])
        # gather patient embeddings to match the OOF arrays via NB07's load_openslidefm_embeddings logic
        df_man = pd.read_csv(manifest_csv)
        df_man['patient_id'] = df_man['slide_id'].str.extract(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', expand=False)
        patient_cancer = df_man.groupby('patient_id')['cancer_code'].first().to_dict()
        records = {}
        def _try_add(npy_path):
            sid = npy_path.stem
            if sid in records: return
            try: emb = np.load(npy_path).astype(np.float32)
            except Exception: return
            if emb.ndim == 1 and emb.shape[0] == 768:
                records[sid] = emb
        for sub in emb_dir.iterdir() if emb_dir.exists() else []:
            if sub.is_dir():
                for npy in sub.glob('*.npy'): _try_add(npy)
        for npy in (emb_dir.glob('*.npy') if emb_dir.exists() else []): _try_add(npy)
        if records:
            import re as _re
            patient_embs = {}
            for sid, emb in records.items():
                m = _re.search(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', sid)
                if m:
                    pid = m.group(1)
                    patient_embs.setdefault(pid, []).append(emb)
            patient_ids = sorted(patient_embs.keys())
            X = np.stack([np.mean(patient_embs[p], axis=0) for p in patient_ids], axis=0)
            cancer_labels = np.array([patient_cancer.get(p, 'UNK') for p in patient_ids])
            cancer_to_idx = {c: i for i, c in enumerate(classes)}
            confidence = probs_oof.max(axis=1) if len(probs_oof) == len(X) else None
            print(f'  fitting UMAP on {len(X)} patients...')
            reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3, random_state=42, metric='cosine')
            U = reducer.fit_transform(X)
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            ax = axes[0]
            for c in classes:
                m = (cancer_labels == c)
                if m.sum() == 0: continue
                ax.scatter(U[m, 0], U[m, 1], s=6, alpha=0.6, label=c)
            ax.set_title('UMAP by cancer type')
            ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
            ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=6, ncol=2)
            ax = axes[1]
            if confidence is not None and len(confidence) == len(U):
                sc = ax.scatter(U[:, 0], U[:, 1], c=confidence, cmap='viridis', s=6, alpha=0.7)
                fig.colorbar(sc, ax=ax, label='prediction confidence')
            else:
                ax.scatter(U[:, 0], U[:, 1], s=6, alpha=0.5, color='#2E86AB')
            ax.set_title('UMAP by prediction confidence')
            ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
            fig.suptitle('Supplementary Figure 1: TCGA slide embeddings UMAP', y=1.02)
            fig.tight_layout()
            save_fig(fig, 'suppfig1_tcga_umap.png')
        else:
            print('  [SKIP] no slide embeddings found in workspace/embeddings/')
    else:
        print('  [SKIP] umap-learn not available')
else:
    print(f'  [SKIP] need {oof_npz} and {manifest_csv} for UMAP figure')

print(f'\n[DONE] NB13 complete. Figures in: {FIG_DIR}')